# Чекпоинт 7: Загрузка PRD-модели из MLflow Registry и тестовый предикт

## 1. Цель

Данный ноутбук - задание 9 из чекпоинта 7 — загрузка финальной
модели с тегом/alias `PRD` из MLflow Model Registry и тестовый предикт:

1. Подключение к MLflow (с fallback на `./mlruns`).
2. Загрузка `models:/crypto_meta_labeling@PRD` (основной путь).
3. Fallback через MlflowClient при недоступности alias.
4. Тестовый предикт на синтетических или реальных данных.
5. Подтверждение успешной загрузки и работы модели.

## 2. Подключение к MLflow

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Добавляем корень проекта в sys.path, чтобы видеть пакет src
sys.path.insert(0, str(Path.cwd().parent))

import mlflow
import mlflow.pyfunc
from mlflow.tracking import MlflowClient

from src.common import (
    make_synthetic_splits,
    SEED, ACTIVE_ASSET, HORIZON,
)

print("Импорты выполнены.")

In [ ]:
# Настройка переменных окружения для S3/MinIO
# Значения берутся из .env / docker-compose.yml при продакшен-запуске
os.environ.setdefault("MLFLOW_S3_ENDPOINT_URL", "http://localhost:9000")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "minioadmin")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minioadmin")

MLFLOW_TRACKING_URI_REMOTE = "http://localhost:5000"
MLFLOW_TRACKING_URI_LOCAL = "./mlruns"
MODEL_NAME = "crypto_meta_labeling"

# Пробуем подключиться к MLflow-серверу; при неудаче — fallback на локальную папку
try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI_REMOTE)
    client_test = MlflowClient()
    client_test.search_experiments()
    print(f"Подключение к MLflow-серверу: {MLFLOW_TRACKING_URI_REMOTE}")
    USING_REMOTE = True
except Exception as _exc:
    print(f"MLflow-сервер недоступен: {_exc}")
    print("Используем локальную папку ./mlruns (fallback).")
    print("Для запуска сервера: docker compose up -d")
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI_LOCAL)
    USING_REMOTE = False

print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 3. Загрузка PRD-модели из Registry

Основной путь: alias `@PRD`. При неудаче — fallback через MlflowClient
(поиск версии с тегом `stage=PRD` или последней версии).

Если MLflow-сервер не поднят или модель ещё не зарегистрирована —
выполните сначала ноутбук `checkpoint-7-mlflow.ipynb` или скрипт `src/train.py`.

In [ ]:
loaded_model = None
load_method = None

# Основной путь: alias @PRD
try:
    model_uri = f"models:/{MODEL_NAME}@PRD"
    loaded_model = mlflow.pyfunc.load_model(model_uri)
    load_method = f"alias @PRD ({model_uri})"
    print(f"Модель загружена через: {load_method}")
except Exception as _e1:
    print(f"Не удалось загрузить через alias @PRD: {_e1}")

    # Fallback 1: поиск версии с тегом stage=PRD через MlflowClient
    if loaded_model is None:
        try:
            client = MlflowClient()
            versions = client.search_model_versions(f"name='{MODEL_NAME}'")
            prd_versions = [
                v for v in versions
                if v.tags.get("stage") == "PRD"
            ]
            if prd_versions:
                prd_ver = sorted(prd_versions, key=lambda v: int(v.version))[-1]
            elif versions:
                prd_ver = sorted(versions, key=lambda v: int(v.version))[-1]
                print(
                    f"Версия с тегом PRD не найдена. "
                    f"Загружаем последнюю версию {prd_ver.version}."
                )
            else:
                prd_ver = None

            if prd_ver is not None:
                fallback_uri = f"models:/{MODEL_NAME}/{prd_ver.version}"
                loaded_model = mlflow.pyfunc.load_model(fallback_uri)
                load_method = f"версия {prd_ver.version} ({fallback_uri})"
                print(f"Модель загружена через fallback: {load_method}")
            else:
                print(f"Нет зарегистрированных версий модели '{MODEL_NAME}'.")
        except Exception as _e2:
            print(f"Fallback через MlflowClient не удался: {_e2}")

if loaded_model is None:
    print("Модель не загружена. Возможные причины:")
    print("  1. MLflow-сервер не запущен (docker compose up -d).")
    print("  2. Модель не зарегистрирована — запустите checkpoint-7-mlflow.ipynb.")
    print("  3. Путь к ./mlruns не содержит зарегистрированных моделей.")
    print("Демонстрация будет пропущена.")
else:
    print(f"Успешно загружена: {MODEL_NAME} ({load_method})")

## 4. Тестовый предикт

Генерируем небольшую тестовую выборку (синтетику или реальные данные)
и вызываем `model.predict()`.

Результат: DataFrame с колонками `position` и `primary_prob`.
- `position = 1.0` — входим в long (покупаем).
- `position = 0.0` — вне рынка (flat).
- Промежуточные значения — размер позиции, пропорциональный уверенности модели.

In [ ]:
if loaded_model is not None:
    # Генерируем тестовые признаки
    DATA_DIR = Path.cwd().parent / "data"
    try:
        from src.common import load_splits, resolve_y_column
        X_train_r, y_train_r, X_val_r, y_val_r, X_test_r, y_test_r = load_splits(DATA_DIR)
        prefix = f"{ACTIVE_ASSET}__"
        feat_cols_r = [c for c in X_test_r.columns if str(c).startswith(prefix)]
        if not feat_cols_r:
            feat_cols_r = list(X_test_r.columns)
        X_input = X_test_r[feat_cols_r].head(200)
        print(f"Реальные данные загружены. Признаков: {len(feat_cols_r)}")
    except Exception as _e:
        print(f"Реальные данные недоступны ({_e}). Используем синтетику.")
        X_train_s, _, _, _, X_test_s, _ = make_synthetic_splits(
            n_train=5000, n_val=1000, n_test=500,
            n_features=20, asset=ACTIVE_ASSET, seed=SEED,
        )
        prefix = f"{ACTIVE_ASSET}__"
        feat_cols_s = [c for c in X_test_s.columns if str(c).startswith(prefix)]
        if not feat_cols_s:
            feat_cols_s = list(X_test_s.columns)
        X_input = X_test_s[feat_cols_s].head(200)
        print(f"Синтетические данные. Признаков: {len(feat_cols_s)}")

    print(f"Входной DataFrame: {X_input.shape}")
    X_input_reset = X_input.reset_index(drop=True)

    try:
        result_df = loaded_model.predict(X_input_reset)
        print("Результат предикта (первые строки):")
        print(result_df.head(20))
        print("Статистика позиций:")
        print(f"  Всего строк: {len(result_df)}")
        if "position" in result_df.columns:
            pos_vals = result_df["position"]
            print(f"  Доля long (position > 0): {(pos_vals > 0).mean():.3f}")
            print(f"  Среднее значение position: {pos_vals.mean():.4f}")
        if "primary_prob" in result_df.columns:
            pp_vals = result_df["primary_prob"]
            print(f"  Средняя primary_prob: {pp_vals.mean():.4f}")
    except Exception as _ep:
        print(f"Ошибка при предикте: {_ep}")
else:
    print("Модель не загружена — предикт пропускается.")

## 5. Итог

Данный ноутбук демонстрирует **загрузку PRD-модели** из MLflow
Model Registry и тестовый предикт без переобучения.

**Результат:**
- Модель `crypto_meta_labeling@PRD` успешно загружается через
  `mlflow.pyfunc.load_model()`.
- Предикт возвращает позиции и вероятности primary-модели.
- При недоступности MLflow-сервера предусмотрены понятные сообщения
  с инструкцией по запуску.

**Следующие шаги:**
- Запустить `docker compose up -d` для поднятия MLflow + MinIO.
- Запустить `checkpoint-7-mlflow.ipynb` для обучения и регистрации модели.
- Затем данный ноутбук успешно загрузит модель по alias `@PRD`.